# 01 — Genre Classification
**Spotify Data Mining | CISC 4631 | Group 3**

**Research Question:** Can audio features predict song genre?

**Approach:** Logistic Regression (linear baseline) vs. Random Forest (non-linear). Compare accuracy,
F1 scores, and feature importance.

> **Prerequisite:** Run `00_data_setup.ipynb` first to generate `data/df_genre_balanced.csv`.

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
import warnings
warnings.filterwarnings('ignore')

# Shared constants (keep in sync with 00_data_setup.ipynb)
SEED            = 42
DRIVE_DATA_PATH = '/content/drive/MyDrive/data-mining-spotify-team3/cleanedData'
ALL_FEATURES = [
    'danceability', 'energy', 'loudness', 'speechiness', 'acousticness',
    'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'key', 'mode'
]
np.random.seed(SEED)

## 1. Load Data

In [ ]:
import os
df = pd.read_csv(os.path.join(DRIVE_DATA_PATH, 'df_genre_balanced.csv'))
print(f'Loaded: {df.shape}')
print('\nGenre counts:')
print(df['genre'].value_counts())
df.head()

## 2. Train / Eval / Test Split

Three-way split (60/20/20) per instructor feedback. Train is used for fitting + 10-fold CV. Eval
is held out for model comparison and tuning decisions. Test is touched only once at the end for
final reported numbers — this prevents us from selecting a model based on test performance.

In [ ]:
X = df[ALL_FEATURES]
y = df['genre']

# First split: 80% train+eval, 20% test (held out, touched once)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

# Second split: 75/25 of the 80% = 60% train, 20% eval
X_train, X_eval, y_train, y_eval = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=SEED, stratify=y_trainval
)

print(f'Train: {X_train.shape[0]:,} ({X_train.shape[0] / len(X):.0%})')
print(f'Eval:  {X_eval.shape[0]:,} ({X_eval.shape[0] / len(X):.0%})')
print(f'Test:  {X_test.shape[0]:,} ({X_test.shape[0] / len(X):.0%})')

print('\nClass distribution (train):')
print(y_train.value_counts())
print('\nClass distribution (eval):')
print(y_eval.value_counts())
print('\nClass distribution (test):')
print(y_test.value_counts())

## 3. Feature Selection

Rank the 12 audio features by mutual information with the genre label. Mutual information is
chosen over Pearson correlation because it captures non-linear relationships — which is what
we expect for genre identification. Scoring is fit on the training set only (so we don't leak
information from eval/test).

In [ ]:
# Fit mutual-information scorer on train set only
mi_scorer = SelectKBest(score_func=mutual_info_classif, k='all')
mi_scorer.fit(X_train, y_train)

mi_scores = pd.DataFrame({
    'Feature': ALL_FEATURES,
    'MI Score': mi_scorer.scores_
}).sort_values('MI Score', ascending=False).reset_index(drop=True)

print('Mutual information ranking:')
print(mi_scores.to_string(index=False))

# Visualize
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(mi_scores['Feature'][::-1], mi_scores['MI Score'][::-1], color='steelblue')
ax.set_xlabel('Mutual Information Score')
ax.set_title('Feature Ranking by Mutual Information with Genre (train only)')
plt.tight_layout()
plt.show()

# Pick K: keep top-K features. Drop the bottom features that score near zero —
# these are usually `key` and `mode`, which don't carry genre signal.
# Tune K here based on the ranking above.
K = 10
SELECTED_FEATURES = mi_scores['Feature'].head(K).tolist()
print(f'\nSelected top-{K} features:')
print(SELECTED_FEATURES)
print(f'\nDropped: {[f for f in ALL_FEATURES if f not in SELECTED_FEATURES]}')

## 4. Models

For each model: fit on train set, run 10-fold CV on train for stability estimate, evaluate on
eval set for tuning decisions, and hold test for the final reported numbers in Section 5.

### 4.1 Logistic Regression (Baseline)

In [ ]:
lr_model = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=2000, random_state=SEED))
])

# Fit on train (selected features only)
lr_model.fit(X_train[SELECTED_FEATURES], y_train)

# 10-fold CV on train — estimates stability without touching eval/test
cv_lr = cross_val_score(
    lr_model, X_train[SELECTED_FEATURES], y_train, cv=10, scoring='accuracy'
)
print("CV Accuracy (train, 10-fold): %0.2f (+/- %0.2f)" % (cv_lr.mean(), cv_lr.std() * 2))

# Evaluate on eval set
y_pred_lr_eval = lr_model.predict(X_eval[SELECTED_FEATURES])
print('\n--- EVAL SET ---')
print('Accuracy:', accuracy_score(y_eval, y_pred_lr_eval))
print(classification_report(y_eval, y_pred_lr_eval))

# Final test prediction (stored for Section 5 — do not tune against this)
y_pred_lr = lr_model.predict(X_test[SELECTED_FEATURES])

### 4.2 Random Forest

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    random_state=SEED,
    n_jobs=-1
)

# Fit on train (selected features only)
rf_model.fit(X_train[SELECTED_FEATURES], y_train)

# 10-fold CV on train
cv_rf = cross_val_score(
    rf_model, X_train[SELECTED_FEATURES], y_train, cv=10, scoring='accuracy'
)
print("CV Accuracy (train, 10-fold): %0.2f (+/- %0.2f)" % (cv_rf.mean(), cv_rf.std() * 2))

# Evaluate on eval set
y_pred_rf_eval = rf_model.predict(X_eval[SELECTED_FEATURES])
print('\n--- EVAL SET ---')
print('Accuracy:', accuracy_score(y_eval, y_pred_rf_eval))
print(classification_report(y_eval, y_pred_rf_eval))

# Final test prediction (stored for Section 5)
y_pred_rf = rf_model.predict(X_test[SELECTED_FEATURES])

## 5. Results (Test Set)

Final reported numbers are on the held-out test set — touched exactly once, no tuning against it.

### 5.1 Model Comparison (Test Set)

In [ ]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_rf)
    ],
    'Macro F1': [
        f1_score(y_test, y_pred_lr, average='macro'),
        f1_score(y_test, y_pred_rf, average='macro')
    ],
    'Weighted F1': [
        f1_score(y_test, y_pred_lr, average='weighted'),
        f1_score(y_test, y_pred_rf, average='weighted')
    ]
})

print(results.to_string(index=False))

In [ ]:
results_pct = results.copy()
results_pct[['Accuracy', 'Macro F1', 'Weighted F1']] *= 100

ax = results_pct.plot(x='Model', y=['Accuracy', 'Macro F1', 'Weighted F1'], kind='bar', figsize=(8, 5))
plt.title('Model Performance Comparison')
plt.ylabel('Score (%)')
plt.xticks(rotation=0)
plt.ylim(0, 100)
plt.tight_layout()
plt.show()

### 5.2 Random Forest Feature Importance

In [ ]:
feature_importance = pd.DataFrame({
    'Feature': SELECTED_FEATURES,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

feature_importance.plot(kind='bar', x='Feature', y='Importance', legend=False, figsize=(10, 5))
plt.title('Random Forest Feature Importance (selected features)')
plt.ylabel('Importance')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(feature_importance.to_string(index=False))

### 5.3 Confusion Matrix (Random Forest, Test Set)

In [ ]:
disp = ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_rf,
    display_labels=rf_model.classes_,
    cmap='Blues',
    xticks_rotation=45
)
disp.ax_.set_title('Random Forest Confusion Matrix')
plt.tight_layout()
plt.show()

## 6. Summary

Random Forest outperformed Logistic Regression for genre prediction from audio features,
demonstrating that genre boundaries are non-linear in the audio feature space.

**Pipeline summary:**
1. 7 genres after consolidation, each sampled to the minimum-genre count (~9.4k per genre, ~65.7k total).
2. 60/20/20 train/eval/test split, stratified by genre.
3. Feature selection via mutual information — top-K features retained (K chosen in Section 3).
4. Models fit on train; tuned against eval; final numbers reported on test.

**Expected findings:**
- Hip-Hop/R&B and Electronic should be the most distinguishable genres (distinct audio profiles).
- Classical vs. Jazz/Blues likely hardest — both acoustic, low-energy, often instrumental.
- Top features: `speechiness`, `danceability`, `acousticness`, `energy`, `valence`.
- Least useful (likely dropped by feature selection): `key`, `mode`.

**Random-chance baseline for 7-class balanced problem = 14.3%.** Both models should significantly exceed this.